# YOLO11n-Seg LKA → SimAM Head — Light/Strong Augmentation, 100e/200e

This notebook trains the LKA → SimAM head configuration on `shrimpdisbd-tigershrimp_mrtudat` version 1. It rebuilds the dataset with a team-approved mixed split before running four independent augmentation/epoch configurations.

Prepared dataset: `tigershrimp_yolo26_mixed_split`. Run cells from top to bottom.


# YOLO11n-seg Nhom C - LKA Head

Architecture experiment for YOLO11n-seg shrimp disease segmentation. Run cells top to bottom. The YAML uses actual YOLO11n channels after width scaling to avoid custom-module channel mismatches.

Training matrix: light augmentation at 100e and 200e, plus strong augmentation at 100e and 200e. Each configuration trains, evaluates, and reports through an isolated run directory.


In [ ]:
# Install dependencies
import importlib.util
import subprocess
import sys


def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])


ensure_package("roboflow")
from importlib.metadata import PackageNotFoundError, version

PINNED_ULTRALYTICS_VERSION = "8.4.61"
try:
    installed_ultralytics = version("ultralytics")
except PackageNotFoundError:
    installed_ultralytics = None
if installed_ultralytics != PINNED_ULTRALYTICS_VERSION:
    if any(name == "ultralytics" or name.startswith("ultralytics.") for name in sys.modules):
        raise RuntimeError("Restart the kernel before pinning Ultralytics for this notebook.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"ultralytics=={PINNED_ULTRALYTICS_VERSION}",
    ])
if version("ultralytics") != PINNED_ULTRALYTICS_VERSION:
    raise RuntimeError("Could not pin the required Ultralytics version.")
ensure_package("yaml", "pyyaml")


In [ ]:
# GPU check
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


In [ ]:
# Team-approved Roboflow download: TigerShrimp / ShrimpDisBD v1 YOLO segmentation.
ROBOFLOW_WORKSPACE = 'lets-try-this'
ROBOFLOW_PROJECT = 'shrimpdisbd-tigershrimp_mrtudat'
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = 'yolo26'

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec('roboflow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'])

from roboflow import Roboflow

ROBOFLOW_API_KEY_DIRECT = ''  # Keep empty. Use a Kaggle Secret named ROBOFLOW_API_KEY.

def get_roboflow_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
        if key:
            return key.strip()
    except Exception:
        pass
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    return os.environ.get('ROBOFLOW_API_KEY', '').strip()

api_key = get_roboflow_api_key()
if not api_key:
    raise RuntimeError('Missing ROBOFLOW_API_KEY. Add it to Kaggle Secrets; do not paste it into the notebook.')

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download(ROBOFLOW_FORMAT)
DATASET_LOCATION = Path(dataset.location)
RAW_YOLO_DATASET_PATH = DATASET_LOCATION
print('Downloaded dataset location:', DATASET_LOCATION)


In [ ]:
## Mixed split: grouped for convention-matched names, stratified random for unmatched names

import hashlib
import os
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import yaml

SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

DATASET_DIR = Path(DATASET_LOCATION)
if not (DATASET_DIR / 'data.yaml').exists():
    yaml_candidates = sorted(DATASET_DIR.rglob('data.yaml'))
    if not yaml_candidates:
        raise FileNotFoundError(f'No data.yaml found below {DATASET_DIR}')
    DATASET_DIR = yaml_candidates[0].parent
base_path = DATASET_DIR
data_yaml_path = DATASET_DIR / 'data.yaml'

SHRIMP_NAME_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)

yaml_content = yaml.safe_load(data_yaml_path.read_text(encoding='utf-8')) or {}
raw_names = yaml_content.get('names', {})
if isinstance(raw_names, list):
    CLASS_NAMES = {i: str(name) for i, name in enumerate(raw_names)}
else:
    CLASS_NAMES = {int(k): str(v) for k, v in raw_names.items()}

def normalize_roboflow_stem(stem):
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)

def parse_shrimp_group_key(image_name):
    stem = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        return None, 'unparsed', None, None
    disease = match.group('disease')
    shrimp_id = match.group('shrimp_id')
    return f'{disease.lower()}::{shrimp_id}', disease, shrimp_id, int(match.group('img_num'))

def image_files_in_split(split):
    directory = base_path / split / 'images'
    if not directory.exists():
        return []
    return sorted(p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

def label_path_for_image(image_path):
    path = image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'
    if path.exists():
        return path
    matches = sorted((image_path.parent.parent / 'labels').glob(f'{image_path.stem}*.txt'))
    return matches[0] if matches else path

def move_image_and_label(image_path, target_split):
    image_dst = base_path / target_split / 'images' / image_path.name
    label_src = label_path_for_image(image_path)
    label_dst = base_path / target_split / 'labels' / f'{image_path.stem}.txt'
    image_dst.parent.mkdir(parents=True, exist_ok=True)
    label_dst.parent.mkdir(parents=True, exist_ok=True)
    if image_path.resolve() != image_dst.resolve():
        if image_dst.exists():
            raise FileExistsError(f'Duplicate image destination: {image_dst}')
        shutil.move(str(image_path), str(image_dst))
    if label_src.exists() and label_src.resolve() != label_dst.resolve():
        if label_dst.exists():
            raise FileExistsError(f'Duplicate label destination: {label_dst}')
        shutil.move(str(label_src), str(label_dst))
    elif not label_dst.exists():
        label_dst.write_text('', encoding='utf-8')

def rebuild_train_pool_from_all_splits():
    all_images = []
    for split in ['train', 'valid', 'test']:
        all_images.extend(image_files_in_split(split))
    for image_path in all_images:
        move_image_and_label(image_path, 'train')
    return image_files_in_split('train')

def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob('**/*.cache'):
        cache_path.unlink()

def image_class_ids(image_path):
    label_path = label_path_for_image(image_path)
    ids = []
    if label_path.exists():
        for line in label_path.read_text(encoding='utf-8').splitlines():
            parts = line.strip().split()
            if parts:
                ids.append(int(float(parts[0])))
    return sorted(set(ids))

def disease_stratum_for_image(image_path):
    ids = image_class_ids(image_path)
    if not ids:
        return 'healthy_empty'
    names = [CLASS_NAMES.get(i, f'class_{i}') for i in ids]
    return '+'.join(name.strip().lower().replace(' ', '_') for name in names)

def split_one_stratum(items, seed, rng=None):
    items = list(items)
    (rng or random.Random(seed)).shuffle(items)
    n = len(items)
    n_train = int(TRAIN_RATIO * n)
    n_valid = int(VAL_RATIO * n)
    if n >= 3:
        n_valid = max(1, n_valid)
        n_train = max(1, n_train)
        if n_train + n_valid >= n:
            n_train = max(1, n - n_valid - 1)
    return {
        'train': items[:n_train],
        'valid': items[n_train:n_train + n_valid],
        'test': items[n_train + n_valid:],
    }

def split_grouped_items(group_items, seed):
    by_stratum = defaultdict(list)
    for group_key, filenames, stratum in group_items:
        by_stratum[stratum].append((group_key, filenames, stratum))
    out = {'train': [], 'valid': [], 'test': []}
    rng = random.Random(seed)
    for stratum, items in sorted(by_stratum.items()):
        pieces = split_one_stratum(sorted(items, key=lambda x: x[0]), seed + 11, rng=rng)
        for split, values in pieces.items():
            out[split].extend(values)
        print(f'Grouped branch {stratum}: {len(pieces["train"])} train groups, {len(pieces["valid"])} valid groups, {len(pieces["test"])} test groups')
    return out

def split_random_items(image_paths, seed):
    by_stratum = defaultdict(list)
    for path in image_paths:
        by_stratum[disease_stratum_for_image(path)].append(path.name)
    out = {'train': [], 'valid': [], 'test': []}
    for stratum, items in sorted(by_stratum.items()):
        pieces = split_one_stratum(sorted(items), seed + 29)
        for split, values in pieces.items():
            out[split].extend(values)
        print(f'Unmatched branch {stratum}: {len(pieces["train"])} train images, {len(pieces["valid"])} valid images, {len(pieces["test"])} test images')
    return out

def mixed_split(seed=42):
    image_paths = rebuild_train_pool_from_all_splits()
    convention_groups = defaultdict(list)
    unmatched = []
    source_meta = {}
    for path in image_paths:
        group_key, filename_disease, shrimp_id, img_num = parse_shrimp_group_key(path.name)
        if group_key is None:
            unmatched.append(path)
            source_meta[path.name] = {'source_subset': 'unmatched_stratified_random', 'naming_convention_matched': False, 'parsed_group_key': f'unparsed::{path.stem}', 'filename_disease': 'unparsed'}
        else:
            convention_groups[group_key].append(path.name)
            source_meta[path.name] = {'source_subset': 'convention_grouped_stratified', 'naming_convention_matched': True, 'parsed_group_key': group_key, 'filename_disease': filename_disease}

    grouped_items = []
    for group_key, filenames in convention_groups.items():
        # Preserve the original baseline's filename-disease stratification for recognized groups.
        strata = Counter(str(source_meta[name]['filename_disease']).lower() for name in filenames)
        grouped_items.append((group_key, sorted(filenames), strata.most_common(1)[0][0]))

    grouped_splits = split_grouped_items(grouped_items, seed)
    random_splits = split_random_items(unmatched, seed)
    assignments = {}
    for branch_splits in [grouped_splits, random_splits]:
        for split, items in branch_splits.items():
            values = items
            if values and isinstance(values[0], tuple):
                names = [name for _, filenames, _ in values for name in filenames]
            else:
                names = values
            for name in names:
                if name in assignments:
                    raise RuntimeError(f'Image assigned twice: {name}')
                assignments[name] = split

    if len(assignments) != len(image_paths):
        missing = sorted({p.name for p in image_paths} - set(assignments))
        raise RuntimeError(f'Mixed split did not assign every image. Missing: {missing[:10]}')

    for name, split in assignments.items():
        move_image_and_label(base_path / 'train' / 'images' / name, split)
    remove_yolo_label_caches(base_path)

    global SOURCE_META
    SOURCE_META = source_meta
    print(f'Mixed split complete: {len(convention_groups)} convention groups and {len(unmatched)} unmatched images.')
    print('Merged image counts:', {split: len(image_files_in_split(split)) for split in ['train', 'valid', 'test']})
    return assignments

mixed_split(SEED)

# The Roboflow export starts with every image in train. Rewrite paths only after
# the approved mixed split has finished moving images and labels.
data_yaml_path = Path(data_yaml_path)
data_yaml_content = yaml.safe_load(data_yaml_path.read_text(encoding='utf-8')) or {}
data_yaml_content['train'] = str((base_path / 'train' / 'images').resolve())
data_yaml_content['val'] = str((base_path / 'valid' / 'images').resolve())
data_yaml_content['test'] = str((base_path / 'test' / 'images').resolve())
data_yaml_path.write_text(
    yaml.safe_dump(data_yaml_content, sort_keys=False, allow_unicode=True),
    encoding='utf-8',
)
print('Updated prepared data.yaml:', data_yaml_path)



In [ ]:
from pathlib import Path

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())
import yaml

if "base_path" not in globals() or "data_yaml_path" not in globals():
    raise RuntimeError("Run the team-approved mixed split cell before this validation cell.")
if "data_yaml_path" not in globals():
    data_yaml_path = str(Path(base_path) / "data.yaml")

config_path = Path(data_yaml_path)
if not config_path.is_file():
    raise FileNotFoundError(f"Prepared data.yaml not found: {config_path}")
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

for split_key, folder in (("train", "train"), ("val", "valid"), ("test", "test")):
    image_dir = Path(base_path) / folder / "images"
    label_dir = Path(base_path) / folder / "labels"
    images = [path for path in image_dir.iterdir() if path.is_file()]
    labels = list(label_dir.glob("*.txt"))
    if not images:
        raise RuntimeError(f"{split_key} split is empty.")
    if len(images) != len(labels):
        raise RuntimeError(
            f"{split_key} image/label mismatch: {len(images)} images, {len(labels)} labels."
        )
    print(f"{split_key}: {len(images)} images, {len(labels)} labels")
print("Verified names:", config.get("names"))
print("Using:", config_path)


In [ ]:
# Runtime patch architecture modules into Ultralytics namespace.
# This avoids fragile source-file text patches and keeps YAML parsing safe.
import torch
import torch.nn as nn

import ultralytics
import ultralytics.nn.modules as nn_modules
import ultralytics.nn.modules.conv as conv_module
import ultralytics.nn.tasks as tasks_module


class SimAM(nn.Module):
    def __init__(self, c1=None, e_lambda=1e-4):
        super().__init__()
        self.e_lambda = e_lambda
        self.activation = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        if n <= 0:
            return x
        x_mu = x - x.mean(dim=[2, 3], keepdim=True)
        denom = 4 * (x_mu.pow(2).sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        y = x_mu.pow(2) / denom + 0.5
        return x * self.activation(y)


class CoordAtt(nn.Module):
    """Channel-preserving Coordinate Attention.

    Compatible with both clean parse_model calls (CoordAtt(c1, reduction))
    and older patched parse_model calls (CoordAtt(c1, c2, reduction)).
    """

    def __init__(self, c1, c2=None, reduction=32):
        super().__init__()
        if c2 is not None and c2 != c1 and reduction == 32:
            # Clean parse_model passes YAML args directly: [c1, reduction].
            reduction = c2
            c2 = c1
        c2 = c1 if c2 is None else c2
        mip = max(8, c1 // reduction)
        self.conv1 = nn.Conv2d(c1, mip, 1, 1, 0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU()
        self.conv_h = nn.Conv2d(mip, c2, 1, 1, 0)
        self.conv_w = nn.Conv2d(mip, c2, 1, 1, 0)
        self.proj = nn.Identity() if c1 == c2 else nn.Conv2d(c1, c2, 1, 1, 0)

    def forward(self, x):
        identity = self.proj(x)
        n, c, h, w = x.size()
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        return identity * self.conv_h(x_h).sigmoid() * self.conv_w(x_w).sigmoid()


class LargeKernelAttention(nn.Module):
    """Channel-preserving LKA. YAML args: [actual_channels]."""

    def __init__(self, c1, kernel_size=5, dilation_kernel_size=7, dilation=3):
        super().__init__()
        self.dw = nn.Conv2d(c1, c1, kernel_size, padding=kernel_size // 2, groups=c1)
        padding = dilation * (dilation_kernel_size - 1) // 2
        self.dw_d = nn.Conv2d(c1, c1, dilation_kernel_size, padding=padding, dilation=dilation, groups=c1)
        self.pw = nn.Conv2d(c1, c1, 1)
        self.gate = nn.Sigmoid()

    def forward(self, x):
        return x * self.gate(self.pw(self.dw_d(self.dw(x))))


for namespace in (conv_module, nn_modules, tasks_module):
    namespace.SimAM = SimAM
    namespace.CoordAtt = CoordAtt
    namespace.LargeKernelAttention = LargeKernelAttention

existing_all = list(getattr(nn_modules, "__all__", []))
for name in ["SimAM", "CoordAtt", "LargeKernelAttention"]:
    if name not in existing_all:
        existing_all.append(name)
nn_modules.__all__ = existing_all

print("Ultralytics:", ultralytics.__version__)
print("Registered modules:", SimAM.__name__, CoordAtt.__name__, LargeKernelAttention.__name__)


In [ ]:
# Write architecture YAML and smoke-build the model.
# Important channel note for YOLO11n:
# layer 16/18 P3 = 64 channels, layer 19/21 P4 = 128 channels, layer 22/24 P5 = 256 channels after width scaling.
from pathlib import Path

from ultralytics import YOLO

EXPERIMENT_KEY = "lka_simam_head_light_strong_100e_200e_new_data_leakage_safe"
EXPERIMENT_NAME = "YOLO11n-seg + LKA->SimAM head (light/strong, 100e/200e, new data)"
YAML_FILENAME = "yolo11n-seg-lka-simam-head.yaml"
YAML_CONTENT = r"""# YOLO11n-seg + LargeKernelAttention->SimAM on P3/P4/P5 head outputs
nc: 2
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

  - [16, 1, LargeKernelAttention, [64]]
  - [23, 1, SimAM, []]
  - [19, 1, LargeKernelAttention, [128]]
  - [25, 1, SimAM, []]
  - [22, 1, LargeKernelAttention, [256]]
  - [27, 1, SimAM, []]
  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]"""

EXPERIMENT_ROOT = WORK_ROOT / "yolov11n_simam_NhomC" / EXPERIMENT_KEY
CFG_DIR = EXPERIMENT_ROOT / "cfg"
CFG_DIR.mkdir(parents=True, exist_ok=True)
yaml_path = CFG_DIR / YAML_FILENAME
yaml_path.write_text(YAML_CONTENT)
print("YAML written to:", yaml_path)

# Smoke build catches channel, concat-size, and custom-module registration errors before training.
smoke_model = YOLO(str(yaml_path))
smoke_model.info()
print("Smoke build OK:", EXPERIMENT_NAME)


In [ ]:
# Evaluation helpers reused from the clean baseline style
import gc
import shutil
import time
from pathlib import Path

import pandas as pd
import yaml
from ultralytics import YOLO

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

COUNT_PENALTY_WEIGHT = 0.05
DISEASE_MISS_PENALTY_WEIGHT = 0.15
HEALTHY_FP_PENALTY_WEIGHT = 0.10
PREDICT_CONF_FOR_COUNT = 0.25

SMOKE_RUN = False  # Set True for a quick one-epoch pass per configuration.
TRAIN_IMGSZ = 320 if SMOKE_RUN else 640
TRAIN_BATCH = 8 if SMOKE_RUN else 16
# Disable early stopping for full 100e/200e comparisons.
TRAIN_PATIENCE = 1 if SMOKE_RUN else 0

LIGHT_AUG_TRAIN_ARGS = {
    # Clean control: mild photometric variation and horizontal flip only.
    # Hidden Albumentations stay disabled so the policy is explicit and reproducible.
    "auto_augment": None,
    "erasing": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "fliplr": 0.5,
    "flipud": 0.0,
    "hsv_h": 0.01,
    "hsv_s": 0.35,
    "hsv_v": 0.20,
    "degrees": 0.0,
    "translate": 0.05,
    "scale": 0.20,
    "shear": 0.0,
    "perspective": 0.0,
    "multi_scale": 0.0,
    "bgr": 0.0,
}

STRONG_AUG_TRAIN_ARGS = {
    "auto_augment": None,
    "erasing": 0.15,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "fliplr": 0.5,
    "flipud": 0.3,
    "hsv_h": 0.05,
    "hsv_s": 0.50,
    "hsv_v": 0.40,
    "degrees": 10.0,
    "translate": 0.10,
    "scale": 0.50,
    "shear": 0.0,
    "perspective": 0.0,
    "multi_scale": 0.0,
    "bgr": 0.0,
}

EXPERIMENTS = [
    {"key": f"{EXPERIMENT_KEY}_light_100e", "name": "YOLO11n-seg + LKA->SimAM head + Light Augmentation (100 epochs)", "policy": "light", "epochs": 100, "train_args": LIGHT_AUG_TRAIN_ARGS},
    {"key": f"{EXPERIMENT_KEY}_light_200e", "name": "YOLO11n-seg + LKA->SimAM head + Light Augmentation (200 epochs)", "policy": "light", "epochs": 200, "train_args": LIGHT_AUG_TRAIN_ARGS},
    {"key": f"{EXPERIMENT_KEY}_strong_100e", "name": "YOLO11n-seg + LKA->SimAM head + Strong Augmentation (100 epochs)", "policy": "strong", "epochs": 100, "train_args": STRONG_AUG_TRAIN_ARGS},
    {"key": f"{EXPERIMENT_KEY}_strong_200e", "name": "YOLO11n-seg + LKA->SimAM head + Strong Augmentation (200 epochs)", "policy": "strong", "epochs": 200, "train_args": STRONG_AUG_TRAIN_ARGS},
]

EXPECTED_RUN_GRID = {('light', 100), ('light', 200), ('strong', 100), ('strong', 200)}
configured_run_grid = {(exp['policy'], exp['epochs']) for exp in EXPERIMENTS}
if configured_run_grid != EXPECTED_RUN_GRID or len(EXPERIMENTS) != 4:
    raise RuntimeError(f'Expected exactly light/strong × 100e/200e; got {EXPERIMENTS}')
print('Scheduled training runs:')
print(pd.DataFrame(EXPERIMENTS)[['policy', 'epochs', 'key']].to_string(index=False))


def disable_ultralytics_albumentations():
    try:
        import ultralytics.data.augment as yolo_augment
    except Exception as exc:
        print(f"Could not patch Ultralytics Albumentations hook: {exc}")
        return

    class NoOpAlbumentations:
        contains_spatial = False

        def __init__(self, *args, **kwargs):
            self.transform = None

        def __call__(self, labels):
            return labels

    yolo_augment.Albumentations = NoOpAlbumentations
    print("Ultralytics Albumentations hook disabled.")


def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob("**/*.cache"):
        cache_path.unlink()


def find_image_for_label(image_dir, label_file):
    stem = Path(label_file).stem
    for ext in IMAGE_EXTENSIONS:
        candidate = Path(image_dir) / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None


def write_data_yaml(dataset_dir, yaml_path, val_dir="valid", test_dir="test"):
    with open(data_yaml_path, "r") as f:
        content = yaml.safe_load(f)
    content["train"] = str(Path(dataset_dir) / "train" / "images")
    content["val"] = str(Path(dataset_dir) / val_dir / "images")
    content["test"] = str(Path(dataset_dir) / test_dir / "images")
    with open(yaml_path, "w") as f:
        yaml.safe_dump(content, f, sort_keys=False)
    return yaml_path


def copy_split_by_label_state(src_dataset, dst_dataset, split, want_labeled):
    src_images = Path(src_dataset) / split / "images"
    src_labels = Path(src_dataset) / split / "labels"
    dst_images = Path(dst_dataset) / split / "images"
    dst_labels = Path(dst_dataset) / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)
    copied = 0
    for label_path in sorted(src_labels.glob("*.txt")):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        if bool(lines) != want_labeled:
            continue
        image_path = find_image_for_label(src_images, label_path.name)
        if image_path is None:
            continue
        shutil.copy2(image_path, dst_images / image_path.name)
        shutil.copy2(label_path, dst_labels / label_path.name)
        copied += 1
    return copied


def make_state_eval_dataset(src_dataset, exp_key, state_name, want_labeled):
    dst = EXPERIMENT_ROOT / f"dataset_{state_name}_eval"
    if dst.exists():
        shutil.rmtree(dst)
    for sub in ["images", "labels"]:
        (dst / "train" / sub).mkdir(parents=True, exist_ok=True)
    copied = {}
    for split in ["valid", "test"]:
        copied[split] = copy_split_by_label_state(src_dataset, dst, split, want_labeled=want_labeled)
    yaml_path = dst / f"data_{state_name}.yaml"
    write_data_yaml(dst, yaml_path)
    print(f"{state_name} eval dataset for {exp_key}: {copied}")
    return dst, yaml_path, copied


def metric_value(metrics, dotted_path, default=float("nan")):
    obj = metrics
    for part in dotted_path.split("."):
        if not hasattr(obj, part):
            return default
        obj = getattr(obj, part)
    try:
        return float(obj)
    except Exception:
        return default


def list_images(images_dir):
    image_paths = []
    for ext in IMAGE_EXTENSIONS:
        image_paths.extend(Path(images_dir).glob(f"*{ext}"))
    return sorted(image_paths)


def count_prediction_errors(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = list_images(images_dir)
    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    box_errors, mask_errors, box_exact, mask_exact = [], [], [], []
    gt_total = pred_box_total = pred_mask_total = disease_images = 0
    disease_box_miss_images = disease_mask_miss_images = 0

    for image_path, result in zip(image_paths, results):
        label_path = Path(labels_dir) / f"{image_path.stem}.txt"
        gt_count = 0
        if label_path.exists():
            gt_count = len([line for line in label_path.read_text().splitlines() if line.strip()])
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        denom = max(1, gt_count)
        box_errors.append(abs(box_count - gt_count) / denom)
        mask_errors.append(abs(mask_count - gt_count) / denom)
        box_exact.append(float(box_count == gt_count))
        mask_exact.append(float(mask_count == gt_count))
        if gt_count > 0:
            disease_images += 1
            disease_box_miss_images += int(box_count == 0)
            disease_mask_miss_images += int(mask_count == 0)
        gt_total += gt_count
        pred_box_total += box_count
        pred_mask_total += mask_count

    return {
        "images": len(image_paths),
        "gt_total": gt_total,
        "pred_box_total": pred_box_total,
        "pred_mask_total": pred_mask_total,
        "box_count_mae": sum(box_errors) / len(box_errors) if box_errors else float("nan"),
        "mask_count_mae": sum(mask_errors) / len(mask_errors) if mask_errors else float("nan"),
        "box_count_exact": sum(box_exact) / len(box_exact) if box_exact else float("nan"),
        "mask_count_exact": sum(mask_exact) / len(mask_exact) if mask_exact else float("nan"),
        "disease_images": disease_images,
        "disease_box_miss_images": disease_box_miss_images,
        "disease_mask_miss_images": disease_mask_miss_images,
        "disease_box_miss_rate": disease_box_miss_images / disease_images if disease_images else float("nan"),
        "disease_mask_miss_rate": disease_mask_miss_images / disease_images if disease_images else float("nan"),
    }


def healthy_false_positive_summary(model, images_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = list_images(images_dir)
    if not image_paths:
        return {
            "healthy_images": 0,
            "healthy_mask_fp_rate": float("nan"),
            "healthy_box_fp_rate": float("nan"),
            "healthy_fp_masks_total": 0,
            "healthy_fp_masks_per_image": float("nan"),
            "healthy_avg_fp_confidence": float("nan"),
        }
    results = model.predict(source=[str(p) for p in image_paths], imgsz=TRAIN_IMGSZ, conf=conf, verbose=False)
    images_with_box_fp = images_with_mask_fp = box_total = mask_total = 0
    confidences = []
    for result in results:
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        if box_count > 0:
            images_with_box_fp += 1
            try:
                confidences.extend([float(v) for v in result.boxes.conf.detach().cpu().tolist()])
            except Exception:
                pass
        if mask_count > 0:
            images_with_mask_fp += 1
        box_total += box_count
        mask_total += mask_count
    n = len(image_paths)
    return {
        "healthy_images": n,
        "healthy_mask_fp_rate": images_with_mask_fp / n,
        "healthy_box_fp_rate": images_with_box_fp / n,
        "healthy_fp_masks_total": mask_total,
        "healthy_fp_masks_per_image": mask_total / n,
        "healthy_avg_fp_confidence": sum(confidences) / len(confidences) if confidences else 0.0,
    }


def read_best_epoch_from_results(run_path):
    results_csv = Path(run_path) / "results.csv"
    if not results_csv.exists():
        return {}
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    mask_col = "metrics/mAP50(M)"
    if mask_col not in df.columns:
        return {"epochs_ran": len(df)}
    best_idx = df[mask_col].idxmax()
    best = df.iloc[best_idx]
    last = df.iloc[-1]
    return {
        "epochs_ran": int(len(df)),
        "best_epoch_by_mask_map50": int(best["epoch"]) if "epoch" in df.columns else int(best_idx + 1),
        "best_val_mask_map50": float(best.get(mask_col, float("nan"))),
        "best_val_mask_map50_95": float(best.get("metrics/mAP50-95(M)", float("nan"))),
        "last_train_seg_loss": float(last.get("train/seg_loss", float("nan"))),
        "last_val_seg_loss": float(last.get("val/seg_loss", float("nan"))),
        "seg_loss_gap_val_minus_train": float(last.get("val/seg_loss", float("nan")) - last.get("train/seg_loss", float("nan"))),
    }


### Required preflight: paths, segmentation labels, and model shapes

This gate runs immediately before any training. It stops the notebook if a split path is invalid, an image has no matching label file, a polygon label is malformed, or the configured model cannot complete a dummy segmentation forward pass.


In [ ]:
# Mandatory preflight gate. Do not bypass this cell before a real training run.
from pathlib import Path
import math

import torch
import yaml
from ultralytics import YOLO

PREFLIGHT_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _preflight_image_dir(value, data_config_path):
    """Resolve a YOLO data.yaml split value to its images directory."""
    if isinstance(value, (list, tuple)):
        if len(value) != 1:
            raise ValueError("This notebook requires exactly one image directory per split.")
        value = value[0]
    image_dir = Path(str(value))
    if not image_dir.is_absolute():
        image_dir = data_config_path.parent / image_dir
    return image_dir.resolve()


def _preflight_validate_dataset(data_config_path):
    data_config_path = Path(data_config_path).resolve()
    if not data_config_path.is_file():
        raise FileNotFoundError(f"Prepared data.yaml was not found: {data_config_path}")

    config = yaml.safe_load(data_config_path.read_text(encoding="utf-8")) or {}
    names = config.get("names", {})
    class_count = len(names) if isinstance(names, (list, tuple, dict)) else 0
    if class_count < 1:
        raise ValueError("data.yaml must define at least one class in `names`.")

    summary = {}
    for split_name in ("train", "val", "test"):
        if split_name not in config:
            raise KeyError(f"data.yaml is missing the `{split_name}` split.")
        image_dir = _preflight_image_dir(config[split_name], data_config_path)
        label_dir = image_dir.parent / "labels"
        if not image_dir.is_dir() or not label_dir.is_dir():
            raise FileNotFoundError(
                f"{split_name}: expected images={image_dir} and labels={label_dir}"
            )

        images = sorted(
            path for path in image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in PREFLIGHT_IMAGE_EXTENSIONS
        )
        if not images:
            raise RuntimeError(f"{split_name}: image split is empty: {image_dir}")

        label_files = sorted(path for path in label_dir.glob("*.txt") if path.is_file())
        image_stems = {path.stem for path in images}
        label_stems = {path.stem for path in label_files}
        missing_labels = sorted(image_stems - label_stems)
        orphan_labels = sorted(label_stems - image_stems)
        if missing_labels or orphan_labels:
            raise RuntimeError(
                f"{split_name}: image/label mismatch; "
                f"missing_labels={missing_labels[:5]}, orphan_labels={orphan_labels[:5]}"
            )

        labeled_images = 0
        polygons = 0
        for label_path in label_files:
            lines = [line.strip() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]
            if lines:
                labeled_images += 1
            for line_number, line in enumerate(lines, start=1):
                tokens = line.split()
                if len(tokens) < 7 or (len(tokens) - 1) % 2 != 0:
                    raise ValueError(
                        f"{split_name}: malformed segmentation polygon at "
                        f"{label_path.name}:{line_number}"
                    )
                try:
                    class_id = int(tokens[0])
                    coordinates = [float(token) for token in tokens[1:]]
                except ValueError as exc:
                    raise ValueError(
                        f"{split_name}: non-numeric label at {label_path.name}:{line_number}"
                    ) from exc
                if not 0 <= class_id < class_count:
                    raise ValueError(
                        f"{split_name}: class id {class_id} out of range at "
                        f"{label_path.name}:{line_number}"
                    )
                if any(not math.isfinite(value) or value < 0.0 or value > 1.0 for value in coordinates):
                    raise ValueError(
                        f"{split_name}: polygon coordinate outside [0, 1] at "
                        f"{label_path.name}:{line_number}"
                    )
                polygons += 1
        summary[split_name] = {
            "images": len(images),
            "empty_labels": len(images) - labeled_images,
            "polygons": polygons,
        }
    return summary


def _preflight_flatten_tensors(value):
    if torch.is_tensor(value):
        yield value
    elif isinstance(value, (tuple, list)):
        for item in value:
            yield from _preflight_flatten_tensors(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from _preflight_flatten_tensors(item)


def run_notebook_preflight(model_specs, expected_attention=None, dummy_imgsz=128):
    """Fail before training on a bad dataset path, label, model build, or tensor shape."""
    if dummy_imgsz % 32:
        raise ValueError("dummy_imgsz must be divisible by 32 for YOLO segmentation.")
    dataset_summary = _preflight_validate_dataset(data_yaml_path)
    print("Dataset preflight PASS:", dataset_summary)

    checked_specs = []
    for model_spec in dict.fromkeys(str(spec) for spec in model_specs):
        model = YOLO(model_spec)
        module = model.model.eval()
        attention_counts = {}
        attention_shape_errors = []
        attention_handles = []
        shape_preserving_names = {
            "CoTEBoundaryLiteGate", "DPCAGate", "LargeKernelAttention", "CoordAtt", "SimAM"
        }

        def _attention_shape_hook(layer_name):
            def hook(_module, inputs, output):
                input_tensor = inputs[0] if inputs else None
                if not torch.is_tensor(input_tensor) or not torch.is_tensor(output):
                    attention_shape_errors.append(f"{layer_name}: non-tensor attention input/output")
                elif input_tensor.ndim != 4 or output.ndim != 4 or tuple(input_tensor.shape) != tuple(output.shape):
                    attention_shape_errors.append(
                        f"{layer_name}: expected shape-preserving BCHW, got "
                        f"{tuple(input_tensor.shape)} -> {tuple(output.shape)}"
                    )
            return hook

        for layer in module.modules():
            layer_name = layer.__class__.__name__
            if layer_name in shape_preserving_names:
                attention_counts[layer_name] = attention_counts.get(layer_name, 0) + 1
                attention_handles.append(layer.register_forward_hook(_attention_shape_hook(layer_name)))
        try:
            device = next(module.parameters()).device
        except StopIteration:
            device = torch.device("cpu")
        dummy = torch.zeros(1, 3, dummy_imgsz, dummy_imgsz, device=device)
        with torch.inference_mode():
            output = module(dummy)
        for handle in attention_handles:
            handle.remove()
        if attention_shape_errors:
            raise RuntimeError(f"{model_spec}: attention shape check failed: {attention_shape_errors}")
        if expected_attention is not None:
            observed_attention = {name: attention_counts.get(name, 0) for name in expected_attention}
            if observed_attention != expected_attention:
                raise RuntimeError(
                    f"{model_spec}: attention module count mismatch; "
                    f"expected={expected_attention}, observed={observed_attention}"
                )
        tensors = list(_preflight_flatten_tensors(output))
        if not tensors:
            raise RuntimeError(f"{model_spec}: model forward returned no tensors.")
        if any(tensor.numel() == 0 or not torch.isfinite(tensor).all().item() for tensor in tensors):
            raise FloatingPointError(f"{model_spec}: model forward produced empty or non-finite tensors.")
        checked_specs.append(model_spec)
        del model, module, dummy, output, tensors
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("Model build/shape preflight PASS:", checked_specs)
    return dataset_summary


In [ ]:

# Hard gate: do not start any training until all paths, polygons and model shapes pass.
run_notebook_preflight([str(yaml_path)], expected_attention={'LargeKernelAttention': 3, 'SimAM': 3})

# Training: four independent augmentation/epoch runs.
from pathlib import Path

from ultralytics import YOLO

EXPERIMENT_ROOT = WORK_ROOT / "yolov11n_simam_NhomC" / EXPERIMENT_KEY
RUNS_DIR = WORK_ROOT / "runs" / "segment"
REPORT_DIR = EXPERIMENT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

dataset_dir = Path(base_path)
remove_yolo_label_caches(dataset_dir)
labeled_eval_dir, labeled_eval_yaml, _ = make_state_eval_dataset(dataset_dir, EXPERIMENT_KEY, "labeled_only", True)
healthy_eval_dir, healthy_eval_yaml, _ = make_state_eval_dataset(dataset_dir, EXPERIMENT_KEY, "healthy_only", False)

trained_runs = []
for experiment in EXPERIMENTS:
    print('\n' + '#' * 90)
    print(f"Starting experiment: {experiment['name']}")
    print('#' * 90)

    run_name = f"yolov11n_simam_{experiment['key']}" + ('_smoke' if SMOKE_RUN else '')
    disable_ultralytics_albumentations()
    yolo = YOLO(str(yaml_path))
    try:
        yolo.load("yolo11n-seg.pt")
        print("Loaded yolo11n-seg pretrained weights.")
    except Exception as exc:
        print("Pretrained load warning:", exc)

    start = time.time()
    yolo.train(
        data=str(data_yaml_path),
        task="segment",
        imgsz=TRAIN_IMGSZ,
        epochs=1 if SMOKE_RUN else experiment['epochs'],
        batch=TRAIN_BATCH,
        patience=TRAIN_PATIENCE,
        seed=42,
        deterministic=True,
        workers=0,
        project=str(RUNS_DIR),
        name=run_name,
        exist_ok=True,
        pretrained=True,
        plots=not SMOKE_RUN,
        verbose=True,
        **experiment['train_args'],
    )
    train_time_min = (time.time() - start) / 60

    run_path = RUNS_DIR / run_name
    best_model_path = run_path / "weights" / "best.pt"
    if not best_model_path.exists():
        raise FileNotFoundError(f"Missing best checkpoint for {experiment['key']}: {best_model_path}")
    trained_runs.append({
        'experiment': experiment['key'],
        'name': experiment['name'],
        'augmentation_policy': experiment['policy'],
        'target_epochs': experiment['epochs'],
        'configured_epochs': 1 if SMOKE_RUN else experiment['epochs'],
        'run_name': run_name,
        'run_path': str(run_path),
        'best_pt': str(best_model_path),
        'train_time_min': round(train_time_min, 2),
    })
    print("Best checkpoint:", best_model_path)
    del yolo
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"Finished {len(trained_runs)} training runs.")


In [ ]:
# Evaluation: full test, labeled-only test, and healthy-negative false positives for every run.
if 'trained_runs' not in globals() or not trained_runs:
    raise RuntimeError('Run the four-configuration training cell before evaluation.')

experiment_results = []
for training_run in trained_runs:
    print('\n' + '-' * 90)
    print(f"Evaluating: {training_run['name']}")
    print('-' * 90)
    best_model = YOLO(training_run['best_pt'])

    full_test = best_model.val(data=str(data_yaml_path), split="test", imgsz=TRAIN_IMGSZ, plots=not SMOKE_RUN, verbose=False)
    labeled_test = best_model.val(data=str(labeled_eval_yaml), split="test", imgsz=TRAIN_IMGSZ, plots=False, verbose=False)

    test_count = count_prediction_errors(best_model, Path(base_path) / "test" / "images", Path(base_path) / "test" / "labels")
    labeled_test_count = count_prediction_errors(best_model, labeled_eval_dir / "test" / "images", labeled_eval_dir / "test" / "labels")
    healthy_test_fp = healthy_false_positive_summary(best_model, healthy_eval_dir / "test" / "images")

    labeled_map50 = metric_value(labeled_test, "seg.map50")
    healthy_aware_score = (
        labeled_map50
        - COUNT_PENALTY_WEIGHT * labeled_test_count["mask_count_mae"]
        - DISEASE_MISS_PENALTY_WEIGHT * labeled_test_count["disease_box_miss_rate"]
        - HEALTHY_FP_PENALTY_WEIGHT * healthy_test_fp["healthy_mask_fp_rate"]
    )

    row = dict(training_run)
    row.update({
        "smoke_run": SMOKE_RUN,
        "full_test_box_map50": metric_value(full_test, "box.map50"),
        "full_test_mask_map50": metric_value(full_test, "seg.map50"),
        "full_test_mask_map50_95": metric_value(full_test, "seg.map"),
        "labeled_test_box_map50": metric_value(labeled_test, "box.map50"),
        "labeled_test_mask_map50": labeled_map50,
        "labeled_test_mask_map50_95": metric_value(labeled_test, "seg.map"),
        "test_gt_instances": test_count["gt_total"],
        "test_pred_masks": test_count["pred_mask_total"],
        "test_mask_count_mae": test_count["mask_count_mae"],
        "labeled_test_gt_instances": labeled_test_count["gt_total"],
        "labeled_test_pred_masks": labeled_test_count["pred_mask_total"],
        "labeled_test_mask_count_mae": labeled_test_count["mask_count_mae"],
        "labeled_test_disease_box_miss_rate": labeled_test_count["disease_box_miss_rate"],
        "labeled_test_disease_mask_miss_rate": labeled_test_count["disease_mask_miss_rate"],
        "healthy_test_images": healthy_test_fp["healthy_images"],
        "healthy_test_mask_fp_rate": healthy_test_fp["healthy_mask_fp_rate"],
        "healthy_test_box_fp_rate": healthy_test_fp["healthy_box_fp_rate"],
        "healthy_test_fp_masks_total": healthy_test_fp["healthy_fp_masks_total"],
        "healthy_test_fp_masks_per_image": healthy_test_fp["healthy_fp_masks_per_image"],
        "healthy_aware_labeled_test_mask_map50": healthy_aware_score,
    })
    row.update(read_best_epoch_from_results(training_run['run_path']))
    experiment_results.append(row)

    partial_df = pd.DataFrame(experiment_results)
    partial_df.to_csv(REPORT_DIR / f"{EXPERIMENT_KEY}_results_partial.csv", index=False)
    del best_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(experiment_results)
summary_csv = REPORT_DIR / f"{EXPERIMENT_KEY}_results_summary.csv"
results_df.to_csv(summary_csv, index=False)
display(results_df)
print("Saved summary for all four runs:", summary_csv)
print('Test metrics are reported for every run and are not used for model selection.')


### Predict the Full Test Set for All Four Configurations

This cell runs inference over every test image for light/strong × 100e/200e. It saves annotated images and YOLO-format prediction labels into isolated directories, one per checkpoint.


In [ ]:
# Predict every test image for every completed light/strong × 100e/200e checkpoint.
from pathlib import Path

if 'results_df' not in globals():
    results_df = pd.read_csv(REPORT_DIR / f"{EXPERIMENT_KEY}_results_summary.csv")

required_columns = {'experiment', 'augmentation_policy', 'target_epochs', 'best_pt'}
missing_columns = required_columns - set(results_df.columns)
if missing_columns:
    raise KeyError(f"Results summary is missing columns: {sorted(missing_columns)}")

expected_prediction_grid = {('light', 100), ('light', 200), ('strong', 100), ('strong', 200)}
completed_prediction_grid = {(str(row['augmentation_policy']), int(row['target_epochs'])) for _, row in results_df.iterrows()}
if completed_prediction_grid != expected_prediction_grid or len(results_df) != 4:
    raise RuntimeError(
        'Prediction requires exactly four completed runs: light/strong × 100e/200e. '
        f'Found: {sorted(completed_prediction_grid)}'
    )

PREDICT_CONF = 0.10
PREDICTION_ROOT = REPORT_DIR / 'predictions'
PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)
test_image_paths = list_images(Path(base_path) / 'test' / 'images')
if not test_image_paths:
    raise RuntimeError('No test images found for prediction.')

prediction_rows = []
for _, run in results_df.sort_values(['augmentation_policy', 'target_epochs']).iterrows():
    experiment_key = str(run['experiment'])
    checkpoint_path = Path(run['best_pt'])
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found for {experiment_key}: {checkpoint_path}")

    print('\n' + '=' * 90)
    print(f"Predicting {len(test_image_paths)} test images: {run['augmentation_policy']} | {run['target_epochs']} epochs")
    print('=' * 90)
    model = YOLO(str(checkpoint_path))
    prediction_results = model.predict(
        source=[str(path) for path in test_image_paths],
        imgsz=TRAIN_IMGSZ,
        conf=PREDICT_CONF,
        save=True,
        save_txt=True,
        save_conf=True,
        project=str(PREDICTION_ROOT),
        name=experiment_key,
        exist_ok=False,
        verbose=False,
    )
    if len(prediction_results) != len(test_image_paths):
        raise RuntimeError(
            f"{experiment_key}: predicted {len(prediction_results)} of {len(test_image_paths)} test images"
        )

    prediction_dir = Path(prediction_results[0].save_dir)
    prediction_rows.append({
        'experiment': experiment_key,
        'augmentation_policy': run['augmentation_policy'],
        'target_epochs': int(run['target_epochs']),
        'checkpoint': str(checkpoint_path),
        'prediction_dir': str(prediction_dir),
        'test_images_predicted': len(prediction_results),
        'confidence_threshold': PREDICT_CONF,
    })
    print(f"Saved annotated test predictions and labels: {prediction_dir}")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

prediction_manifest_df = pd.DataFrame(prediction_rows)
if len(prediction_manifest_df) != 4 or not (prediction_manifest_df['test_images_predicted'] == len(test_image_paths)).all():
    raise RuntimeError('Prediction manifest is incomplete; expected all test images for all four configurations.')
prediction_manifest_csv = PREDICTION_ROOT / 'prediction_manifest.csv'
prediction_manifest_df.to_csv(prediction_manifest_csv, index=False)
display(prediction_manifest_df)
print(f"Saved full-test prediction manifest: {prediction_manifest_csv}")
